In [10]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from pcm_pix.io import load_materials

m = load_materials(CFG, base_dir="data")

sb2se3_am = m.sb2se3_am
sb2se3_cr = m.sb2se3_cr
gst_am = m.gst_am
gst_cr = m.gst_cr

logger.info("materials loaded: sb2se3_am=%s, sb2se3_cr=%s", len(sb2se3_am), len(sb2se3_cr))

2026-03-03 12:17:19,917 | INFO | materials loaded: sb2se3_am=171, sb2se3_cr=171


In [12]:
CFG["mesh_names"] = [
    "Sb2Se3 - amorphous_mesh_sbse_2025.txt",
    "Sb2Se3 - crystal_mesh12_sbse_2025.txt",
]

In [14]:
import importlib
import pcm_pix.features as f
importlib.reload(f)

<module 'pcm_pix.features' from '/home/slava/КВАНТЭЛ/GIT/PCM_pix/pcm_pix/features.py'>

In [16]:
from pcm_pix.features import load_mesh_tables, make_nn_dataset

mesh_tables = load_mesh_tables(CFG, base_dir="data")
ds = make_nn_dataset(mesh_tables, wl=CFG["wl"])

logger.info("dataset: df=%s, data_0=%s, data_1=%s", len(ds.df), len(ds.data_0), len(ds.data_1))
logger.info("X_0=%s y_0=%s | X_1=%s y_1=%s", ds.X_0.shape, ds.y_0.shape, ds.X_1.shape, ds.y_1.shape)

2026-03-03 12:24:20,159 | INFO | dataset: df=98390, data_0=60360, data_1=38030
2026-03-03 12:24:20,160 | INFO | X_0=(60360, 3) y_0=(60360, 4) | X_1=(38030, 3) y_1=(38030, 4)


In [19]:
from pcm_pix.nn_surrogate import train_or_load_surrogates, get_device

CFG["device"] = get_device()
CFG["epochs"] = 400   # для первой проверки можно 200-500, чтобы быстро
CFG["lr"] = 1e-3

sur0, sur1 = train_or_load_surrogates(ds, run, CFG, force_train=False)
logger.info("surrogates ready")

2026-03-03 12:31:31,440 | INFO | training surrogates (device=cpu, epochs=400, lr=0.001)


2026-03-03 12:31:32,084 | INFO | epoch=0 train_loss=0.229170 test_loss=0.207266
2026-03-03 12:32:10,489 | INFO | epoch=200 train_loss=0.047483 test_loss=0.047535
2026-03-03 12:32:42,162 | INFO | epoch=399 train_loss=0.034855 test_loss=0.034993
2026-03-03 12:32:42,297 | INFO | epoch=0 train_loss=0.269578 test_loss=0.240919
2026-03-03 12:32:56,856 | INFO | epoch=200 train_loss=0.063923 test_loss=0.062786
2026-03-03 12:33:13,975 | INFO | epoch=399 train_loss=0.041948 test_loss=0.040982
2026-03-03 12:33:13,981 | INFO | saved models to outputs/PCM_bagel_2025/models
2026-03-03 12:33:13,982 | INFO | am metrics={'train_loss': 0.034855280071496964, 'test_loss': 0.03499330207705498} | cr metrics={'train_loss': 0.041947852820158005, 'test_loss': 0.0409817211329937}
2026-03-03 12:33:13,984 | INFO | surrogates ready


In [26]:
from pcm_pix.nn_surrogate import Net
from pcm_pix.nn_surrogate import load_legacy_surrogates

CFG["legacy_dir"] = "data"
CFG["device"] = "cpu"

sur0, sur1 = load_legacy_surrogates(CFG, device=CFG["device"])
logger.info("legacy surrogates loaded")

2026-03-03 12:43:31,773 | INFO | legacy surrogates loaded


In [ ]:
import numpy as np

PRESETS = {
    "pos_2026_03_03_example": np.array([999.9999832237819, 984.3026291343397, 958.2542604775822, 936.6180434251005, 
                                        993.4814923872052, 999.9999543918377, 950.8023314568704, 999.7122801728465, 
                                        812.6419505503554, 900.387276453811, 954.4390294470007, 650.2800698156437, 
                                        662.5287955357173, 681.2858693125721, 700.083406735213, 726.3045463730607, 
                                        931.6468661009544, 770.9736892629372, 819.773820550387, 569.5143584777072, 
                                        520.9010616227479, 480.96992943410584, 314.16505765104455, 311.1638267139905, 
                                        310.6361359995624, 323.64108002976633, 469.1378222786072, 530.8057496399709, 
                                        519.5742333256374, 690.5658277235785, 50.05271853544764, 7.789435407260271, 
                                        37.236000971268254, 5.051730929403151, 3.8741672878189197, 0.9047503198059405, 
                                        0.9887957678343591]),
}
from pcm_pix.optimize_pso import save_solution

for name, pos in PRESETS.items():
    save_solution(run, name=name, pos=pos, cost=None, meta={"source": "old_notebook"})

2026-03-03 13:02:46,736 | INFO | saved solution pos_2026_03_03_example.json


In [ ]:
from pcm_pix.optimize_pso import run_pso_until, run_de_full

# 1) PSO как в ноутбуке (порог + рестарты)
CFG["pso_threshold"] = 4
CFG["pso_max_restarts"] = 20
CFG["pso_n_particles"] = 3000
CFG["pso_iters"] = 1000
CFG["pso_c1"] = 0.5
CFG["pso_c2"] = 0.3
CFG["pso_w"] = 0.9

best_pso = run_pso_until(sur0, sur1, CFG, run)

In [ ]:
# 2) DE как в ноутбуке (init_ar + vectorized + callback)
CFG["de_init_mode"] = "init_ar"
CFG["de_init_N"] = 1000
CFG["de_mutation"] = 0.99
CFG["de_recombination"] = 0.1
CFG["de_maxiter"] = 1000000
CFG["de_popsize"] = 5000
CFG["de_tol"] = 1e-12
CFG["de_atol"] = 1e-12
CFG["de_polish"] = True
CFG["de_callback_every"] = 1000
CFG["de_use_linear_constraint"] = True  # если хочешь именно constraints-ветку — поставь True

best = run_de_full(sur0, sur1, CFG, run, pos=best_pso.pos)
best

In [3]:
from pcm_pix.run import start_run

run = start_run(outputs_dir="outputs", run_name=CFG["run_name"])
logger = run.logger
logger.info("main.ipynb started")

2026-03-03 12:07:56,122 | INFO | run_dir=/home/slava/КВАНТЭЛ/GIT/PCM_pix/outputs/PCM_bagel_2025
2026-03-03 12:07:56,127 | INFO | main.ipynb started


In [4]:
# TODO: data = load_inputs(CFG)

In [5]:
# TODO: sur0, sur1 = train_or_load(...)

In [6]:
# TODO: best = optimize(...)
# TODO: export_gds(...)